In [21]:
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from datasets.arrow_dataset import Dataset as HFDataset

from torchvision.transforms import v2

from torch import Tensor
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from vit import *
from eval import *

from IPython.display import clear_output

import sys


train_dataset = load_dataset(
    "ILSVRC/imagenet-1k",
    split="train",
)

val_dataset = load_dataset(
    "ILSVRC/imagenet-1k",
    split="validation",
)

test_dataset = load_dataset(
    "ILSVRC/imagenet-1k",
    split="test",
)




Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/267 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

... (4 outputs left out)

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [22]:
class ImageNetDataset(Dataset):
    def __init__(self, dataset: HFDataset, transform: v2.Transform = None) -> None:
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx: int) -> tuple[Tensor, int]:
        sample = self.dataset[idx]

        image = sample["image"]
        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

eval_transform = v2.Compose([
    v2.ToImage(),

    # preserves aspect ratio, addresses varying image resolutions
    v2.Resize(size=256, interpolation=v2.InterpolationMode.BICUBIC),
    v2.CenterCrop(size=224),

    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

train_transform = v2.Compose([
    v2.ToImage(),

    v2.RandomResizedCrop( # defaults
        size=224,
        scale=(0.08, 1.0),
        ratio=(3/4, 4/3),
        interpolation=v2.InterpolationMode.BICUBIC
    ),
    v2.RandomHorizontalFlip(p=0.5),

    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])

clean_train_dataset = ImageNetDataset(train_dataset, eval_transform)
train_dataset = ImageNetDataset(train_dataset, train_transform)

val_dataset = ImageNetDataset(val_dataset, eval_transform)
test_dataset = ImageNetDataset(test_dataset, eval_transform)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers=16,
    pin_memory=True
)

clean_train_dataloader = DataLoader(
    clean_train_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False,
    num_workers=16,
    pin_memory=True
)
